In [ ]:
import itertools
from dataclasses import dataclass
from pathlib import Path

import matplotlib as mpl
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import multiprocess as mp
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from scipy.ndimage import center_of_mass

from discrete_simulation import DiscreteSimulation

mpl.rcParams.update(mpl.rcParamsDefault)
plt.style.use('bmh')
mpl.rcParams['font.family'] = 'DejaVu Serif'
mpl.rcParams['font.size'] = 24
mpl.rcParams['axes.linewidth'] = 2

ticks_dir = 'in'
axis_label_size = 18

Path('plots').mkdir(exist_ok=True)

In [ ]:
@dataclass
class SimulationResult:
    m_c: int
    fidelities: np.ndarray


d = 16
m_i = 16
m_f = 2
alpha = 4
fidelity_cut = 0
resolution = 100

gamma_loss_values = np.logspace(-4, -2, resolution)
gamma_phi_values = np.logspace(-3, -1, resolution)
m_c_values = [16, 8, 4, 2]
process_count = min(50, mp.cpu_count())


def run_sweep(m_c):
    simulation = DiscreteSimulation(d, m_i, m_c, alpha=alpha, m_f=m_f)
    parameters = itertools.product(
        gamma_loss_values,
        gamma_phi_values,
        [fidelity_cut],
    )
    with mp.Pool(process_count) as pool:
        fidelity_probability_pairs = pool.starmap(
            simulation.average_fidelity,
            parameters,
        )

    fidelities = np.fromiter(
        (fidelity for fidelity, _ in fidelity_probability_pairs),
        dtype=float,
        count=resolution**2,
    )
    return SimulationResult(m_c=m_c, fidelities=fidelities)


results = [run_sweep(m_c) for m_c in m_c_values]

In [ ]:
def get_color_bar():
    ylorrd = mpl.colormaps['YlOrRd'].resampled(256)
    n_transition = 50
    first_color = ylorrd(0)

    colors_list = []
    for i in range(n_transition):
        fraction = i / n_transition
        red = 1 - (1 - first_color[0]) * fraction
        green = 1 - (1 - first_color[1]) * fraction
        blue = 1 - (1 - first_color[2]) * fraction
        colors_list.append((red, green, blue, 1))

    colors_list.extend(
        ylorrd(i / 256) for i in range(256 - n_transition)
    )
    return LinearSegmentedColormap.from_list(
        'WhiteToYlOrRd',
        colors_list,
        N=256,
    )

In [ ]:
gamma_phi_grid, gamma_loss_grid = np.meshgrid(
    gamma_phi_values,
    gamma_loss_values,
)
infidelity_arrays = [
    1 - result.fidelities.reshape(resolution, resolution)
    for result in results
]
positive_infidelities = np.concatenate(
    [values[values > 0] for values in infidelity_arrays]
)
vmin = positive_infidelities.min()
vmax = 0.5

num_rows = (len(results) + 1) // 2
fig, axs = plt.subplots(num_rows, 2, figsize=(12, 5 * num_rows))

for i, (ax, result, infidelity_array) in enumerate(
    zip(axs.flatten(), results, infidelity_arrays)
):
    fidelity_array = 1 - infidelity_array
    pcm = ax.pcolormesh(
        gamma_phi_grid,
        gamma_loss_grid,
        infidelity_array,
        cmap=get_color_bar(),
        norm=colors.LogNorm(vmin=vmin, vmax=vmax),
        shading='auto',
        rasterized=True,
    )
    cbar = fig.colorbar(pcm, ax=ax)
    cbar.ax.invert_yaxis()
    cbar.ax.tick_params(labelsize=axis_label_size)

    if i % 2 == 1:
        cbar.set_label('Infidelity', size=20)
        ax.yaxis.set_tick_params(which='major', labelleft=False)
    else:
        ax.set_ylabel(r'$\gamma_l$')

    ax.xaxis.set_tick_params(
        which='major', size=6, width=1, direction=ticks_dir,
        labelsize=axis_label_size,
    )
    ax.xaxis.set_tick_params(
        which='minor', size=3, width=1, direction=ticks_dir,
        labelsize=axis_label_size,
    )
    ax.yaxis.set_tick_params(
        which='major', size=6, width=1, direction=ticks_dir,
        labelsize=axis_label_size,
    )
    ax.yaxis.set_tick_params(
        which='minor', size=3, width=1, direction=ticks_dir,
        labelsize=axis_label_size,
    )
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.grid(None)

    if i // 2 == num_rows - 1:
        ax.set_xlabel(r'$\gamma_\phi$')
    else:
        ax.xaxis.set_tick_params(which='major', labelbottom=False)

    ax.text(
        0.05,
        0.95,
        f'$m_c={result.m_c}$',
        transform=ax.transAxes,
        fontsize=30,
        va='top',
        color='black',
    )

    fidelity_contours = ax.contour(
        gamma_phi_grid,
        gamma_loss_grid,
        fidelity_array,
        colors='dimgray',
        levels=[0, 0.7, 0.9, 0.99, 0.999, 1],
        linewidths=1,
    )
    labels = ax.clabel(
        fidelity_contours,
        fidelity_contours.levels,
        inline=True,
        inline_spacing=0,
        fontsize=10,
    )
    label_positions = [label.get_position() for label in labels]
    for label in labels:
        label.remove()
    ax.clabel(
        fidelity_contours,
        fidelity_contours.levels,
        inline=True,
        inline_spacing=0,
        fontsize=15,
        manual=label_positions,
    )

fig.align_labels()
plt.tight_layout()
plt.subplots_adjust(wspace=0.05, hspace=0.1)
plt.savefig(f'plots/different_m_c,{alpha=}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fidelity_arrays = [
    result.fidelities.reshape(resolution, resolution)
    for result in results
]
stacked_fidelities = np.stack(fidelity_arrays)
best_fidelities = np.max(stacked_fidelities, axis=0)
best_result_indices = np.argmax(stacked_fidelities, axis=0)
gamma_phi_grid, gamma_loss_grid = np.meshgrid(
    gamma_phi_values,
    gamma_loss_values,
)

fig, ax = plt.subplots(figsize=(12, 10))
pcm = ax.pcolormesh(
    gamma_phi_grid,
    gamma_loss_grid,
    1 - best_fidelities,
    cmap='RdPu',
    norm=colors.LogNorm(),
    shading='auto',
    rasterized=True,
)
cbar = fig.colorbar(pcm, ax=ax, label='Infidelity')
cbar.set_label('Infidelity', size=20)
cbar.ax.invert_yaxis()
cbar.ax.tick_params(labelsize=16)

ax.contour(
    gamma_phi_grid,
    gamma_loss_grid,
    best_result_indices,
    colors='black',
    levels=np.arange(-0.5, len(results) + 0.5, 1),
    linewidths=1,
)

ax.xaxis.set_tick_params(
    which='major', size=14, width=1, direction=ticks_dir,
    labelsize=axis_label_size,
)
ax.xaxis.set_tick_params(
    which='minor', size=7, width=1, direction=ticks_dir,
    labelsize=axis_label_size,
)
ax.yaxis.set_tick_params(
    which='major', size=14, width=1, direction=ticks_dir,
    labelsize=axis_label_size,
)
ax.yaxis.set_tick_params(
    which='minor', size=7, width=1, direction=ticks_dir,
    labelsize=axis_label_size,
)
ax.grid(None)

for result_index in np.unique(best_result_indices):
    mask = best_result_indices == result_index
    y_center, x_center = center_of_mass(mask)
    ax.text(
        gamma_phi_values[round(x_center)],
        gamma_loss_values[round(y_center)],
        f'$m_c={results[result_index].m_c}$',
        fontsize=30,
        ha='center',
        va='center',
        color='black',
    )

ax.set_xlim(gamma_phi_values.min(), gamma_phi_values.max())
ax.set_ylim(gamma_loss_values.min(), gamma_loss_values.max())
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_ylabel(r'$\gamma_l$')
ax.set_xlabel(r'$\gamma_\phi$')

plt.savefig(f'plots/best_m_c,{alpha=}.svg', bbox_inches='tight')
plt.show()